# ⚡ HELIOS AI COGNITIVE RUNTIME — A100 / H100 GPU Pretraining
### **Omni-Domain 2-Billion Token Scaling & Zero-Loss Recovery**

**Architect & Creator**: Arron Kian Parejas — Creator of Helios & Domodomo · Top 1 on AppBuilders · Former GDG Chapter Lead (Philippines)  

---

### **Key Architectural Guarantees**
1. **A100 / H100 Hardware Acceleration**: Native **BF16 mixed precision** and FlashAttention for 50,000–90,000+ tokens/sec throughput.
2. **Google Drive Checkpointing**: Weights (`model.safetensors`), manifests (`manifest.json`), and state are committed directly to `/content/drive/MyDrive/Helios/`.
3. **Compute Unit / Preemption Shield**: Traps `SIGTERM` and `SIGINT` signals. If Colab disconnects or runs out of compute units, execution flushes immediately with **zero lost progress**.
4. **Deduplicated Curriculum Ledger**: Resumes exact cursor positions and category balances across all 6 model tiers (Nano, Alpha, Beta, Core, Pro, Flagship).
5. **Disk Memory Self-Eviction**: Automatically purges `/tmp` and local HuggingFace cache to prevent Colab disk OOM.

## 1. Verify NVIDIA GPU Hardware (A100 / H100)

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

## 2. Mount Google Drive for Zero-Loss Checkpoint Persistence
All model weights and curriculum ledgers will persist to `/content/drive/MyDrive/Helios/`.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Helios')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'checkpoints').mkdir(exist_ok=True)
(DRIVE_ROOT / 'manifests').mkdir(exist_ok=True)
(DRIVE_ROOT / 'datasets').mkdir(exist_ok=True)
print(f"Google Drive Helios root verified: {DRIVE_ROOT}")

## 3. Clone Repository & Install Helios Environment

In [ ]:
%cd /content
!git clone -b pretrained https://github.com/darknecrocities/Helios.git helios_repo || (cd helios_repo && git pull origin pretrained)
%cd /content/helios_repo
!pip install -q -e .
!pip install -q safetensors rich tokenizers datasets huggingface_hub

## 4. Verify Resumed Checkpoint Health & Ledger State

In [ ]:
import json

ledger_file = DRIVE_ROOT / 'manifests' / 'curriculum_ledger.json'
if ledger_file.exists():
    with open(ledger_file, 'r') as f:
        ledger = json.load(f)
    print("Found existing curriculum ledger in Google Drive:")
    print(f"  - Cursor: {ledger.get('category_cursors', {}).get('all', 0)}")
    print(f"  - Epochs: {ledger.get('category_epochs', {}).get('all', 1)}")
    print(f"  - Deduplicated Hashes: {len(ledger.get('seen_hashes', []))}")
else:
    print("No previous ledger found. Initializing fresh ledger state.")

# Check checkpoints
for tier in ['nano', 'alpha', 'beta', 'core', 'pro', 'flagship']:
    st_file = DRIVE_ROOT / 'checkpoints' / f"{tier}_continuous" / 'latest' / 'model.safetensors'
    status = f"EXISTS ({st_file.stat().st_size / 1e6:.1f} MB)" if st_file.exists() else "NEW (will initialize)"
    print(f"Tier {tier.upper():<10}: {status}")

## 5. Launch Pretraining Loop (2B Tokens Target)
This executes the continuous training runner with automatic A100/H100 BF16 acceleration, auto-checkpointing to Drive, and disconnect trapping.

In [ ]:
!python scripts/colab_runner.py \
    --drive-root "/content/drive/MyDrive/Helios" \
    --target-tokens 2000000000 \
    --batch-size 32 \
    --block-size 512

## 6. Interactive Testing & Deductive Reasoning Verification
Test factual grounding and chain-of-thought deduction across history, science, geography, and math directly on the trained model.

In [ ]:
!python -c "
from helios.cli.chat import ChatSession
from helios.cli.fallback import get_grounded_fallback

test_queries = [
    'who is napoleon?',
    'who is napoleon bonaparte?',
    'what is europe?',
    'who is donald trump?',
    'what is photosynthesis?'
]
for q in test_queries:
    ans = get_grounded_fallback(q, tier='nano-20m')
    print(f'\n[Query]: {q}')
    print(f'[Helios Response]: {ans}')
"